In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from keras import initializers
import ast

Datasets = []
PREDICTORS = ["PwmD", "PwmE", "Theta"]   
TARGET_INT = ["Theta", "X", "Y"]  
TARGET = ["DeltaTheta", "DeltaX", "DeltaY",]
SETS = [
    "Train_1",
    "Train_2",
    "Test_1",
    "Test_2",
    "Test_3",
    "Val",
    "LSG1",
    "LSG2"
]
     
TIME_STEPS = 3
TS = 0.07

In [2]:
for i in range(4):
    Dataset = pd.read_excel(f"./../../RotedData/Data.xlsx", f"D{i+1}")   
    Datasets.append(Dataset)

for i in range(4):   
    Dataset = pd.read_csv(f"./../../Data/Data{i + 1}.csv")  
    Datasets.append(Dataset)
    
    
for i in range(len(Datasets)):
    Dataset = Datasets[i].copy()

    for var in TARGET_INT:
        Dataset[f"Delta{var}"] = (Dataset[var].shift(-1) - Dataset[var]) / TS

    Dataset = Dataset.dropna(subset=[f"Delta{var}" for var in TARGET_INT])

    Datasets[i] = Dataset

In [3]:
results = [] 
for target in TARGET_INT:
    result = pd.read_excel(f"./models/{target}.xlsx")
    results.append(result)

In [4]:
NormDatasets = []

INPUT_SCALER = StandardScaler()

OUT_SCALERS = {
    "DeltaTheta": StandardScaler(),
    "DeltaX": StandardScaler(),
    "DeltaY": StandardScaler()
}

# ======================
# Train 1
# ======================
Train1 = Datasets[0].copy()

Train1[PREDICTORS] = INPUT_SCALER.fit_transform(Train1[PREDICTORS])

for t in TARGET:
    Train1[[t]] = OUT_SCALERS[t].fit_transform(Train1[[t]])

NormDatasets.append(Train1)

# ======================
# Train 2
# ======================
Train2 = Datasets[1].copy()

Train2[PREDICTORS] = INPUT_SCALER.transform(Train2[PREDICTORS])

for t in TARGET:
    Train2[[t]] = OUT_SCALERS[t].transform(Train2[[t]])

NormDatasets.append(Train2)

# concatena treino
Train = pd.concat([Train1, Train2], ignore_index=True)

# ======================
# Testes + Val
# ======================
for i in range(6):

    CurrentTestDataset = Datasets[i + 2].copy()

    CurrentTestDataset[PREDICTORS] = INPUT_SCALER.transform(
        CurrentTestDataset[PREDICTORS]
    )

    for t in TARGET:
        CurrentTestDataset[[t]] = OUT_SCALERS[t].transform(
            CurrentTestDataset[[t]]
        )

    NormDatasets.append(CurrentTestDataset)

In [5]:
import pandas as pd

def PickModels(df, target, tr1=0.0, tr2=0.0, v1=0.0, t3=0.0):

    r2_tr1 = f"R2_Train_1_{target}"
    r2_tr2 = f"R2_Train_2_{target}"
    r2_val = f"R2_Val_{target}"
    r2_t3 = f"R2_Test_3_{target}"

    # filtro mínimo
    filtered = df[
        (df[r2_tr1] > tr1) &
        (df[r2_tr2] > tr2) &
        (df[r2_t3] > t3) &
        (df[r2_val] > v1)
    ]

    if filtered.empty:
        print("Nenhum modelo satisfaz os critérios.")
        return None

    # 🔹 melhor para cada métrica
    best_tr1 = filtered.sort_values(r2_tr1, ascending=False).iloc[0]
    best_tr2 = filtered.sort_values(r2_tr2, ascending=False).iloc[0]
    best_val = filtered.sort_values(r2_val, ascending=False).iloc[0]
    best_t3 = filtered.sort_values(r2_t3, ascending=False).iloc[0]

    selected = pd.DataFrame([best_tr1, best_tr2, best_val, best_t3]).drop_duplicates()

    cols = ["model", "Neurons", "reg",  r2_tr1, r2_tr2, r2_val, r2_t3]

    table = selected[cols].copy()

    # 🔹 calcular média das métricas
    mean_values = table[[r2_tr1, r2_tr2, r2_val, r2_t3]].mean()

    mean_row = pd.DataFrame([{
        "model": "MEAN",
        "Neurons": "-",
        r2_tr1: mean_values[r2_tr1],
        r2_tr2: mean_values[r2_tr2],
        r2_val: mean_values[r2_val],
        r2_t3: mean_values[r2_t3]
    }])

    table_mean = pd.concat([table, mean_row], ignore_index=True)

    display(table_mean)

    return table

In [6]:
models = []
models.append(PickModels(results[0], "Theta", tr1=0.8, tr2=0.6, v1=0.6, t3=0.6))

,model,Neurons,reg,R2_Train_1_Theta,R2_Train_2_Theta,R2_Val_Theta,R2_Test_3_Theta
0,model_arch16-8-4_r0.01_seed8716,"[16, 8, 4]",0.01,0.901765,0.69352,0.751616,0.773393
1,MEAN,-,NaN,0.901765,0.69352,0.751616,0.773393


In [7]:
models.append(PickModels(results[1], "X", tr1=0.0, tr2=0.0, v1=0.0, t3=0.0))

,model,Neurons,reg,R2_Train_1_X,R2_Train_2_X,R2_Val_X,R2_Test_3_X
0,model_arch32-16_r0.01_seed4026,"[32, 16]",0.01,0.838725,0.81863,0.907652,0.337397
1,MEAN,-,NaN,0.838725,0.81863,0.907652,0.337397


In [8]:
models.append(PickModels(results[2], "Y", tr1=0.6, tr2=0.6, v1=0.6, t3=0.6))

,model,Neurons,reg,R2_Train_1_Y,R2_Train_2_Y,R2_Val_Y,R2_Test_3_Y
0,model_arch16-8_r0.01_seed8547,"[16, 8]",0.01,0.776836,0.959904,0.760223,0.714847
1,model_arch64_r0.01_seed5123,[64],0.01,0.603053,0.970937,0.731114,0.890678
2,model_arch16-8_r0.01_seed174,"[16, 8]",0.01,0.750018,0.919498,0.808920,0.867590
3,MEAN,-,NaN,0.709969,0.950113,0.766752,0.824371


In [9]:
from keras.saving import register_keras_serializable
import tensorflow as tf
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

@register_keras_serializable()
class PINN(tf.keras.Model):

    def __init__(self, architecture, initializer, regularizer):
        super(PINN, self).__init__()

        self.rnn_layers = []

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        return_sequences=True,
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )
            else:
                self.rnn_layers.append(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation='tanh',
                        kernel_initializer=initializer,
                        kernel_regularizer=regularizer,
                        recurrent_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        # camada densa final
        self.out_layer = tf.keras.layers.Dense(
            1,
            activation="linear",
            kernel_initializer=initializer,
            kernel_regularizer=regularizer,
            bias_regularizer=regularizer
        )

    def call(self, inputs):

        x = inputs

        for rnn in self.rnn_layers:
            x = rnn(x)

        return self.out_layer(x)

In [10]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

def CreateSequences(input_data, target_data, timesteps):

    X_seq, Y_seq = [], []

    for i in range(timesteps, len(input_data)):
        X_seq.append(input_data.iloc[i-timesteps:i].values)
        Y_seq.append(target_data.iloc[i].values)

    return np.array(X_seq), np.array(Y_seq)


class EnsembleNets():

    def __init__(self, predictors, targets, int_targets, out_scaler):

        self.models = []
        self.predictors = predictors
        self.targets = targets
        self.target_int = int_targets
        self.input_size = len(predictors)
        self.output_size = len(targets)
        self.out_scaler = out_scaler


    def SetModels(self, architecture, Wf, r):
        regularizer = tf.keras.regularizers.l2(r)

        model = tf.keras.Sequential()
        # model.add(tf.keras.layers.Input(shape=(TIME_STEPS, self.input_size)))

        for i, units in enumerate(architecture):

            if i < len(architecture) - 1:
                model.add(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation="tanh",
                        return_sequences=True,
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

            else:
                model.add(
                    tf.keras.layers.SimpleRNN(
                        units,
                        activation="tanh",
                        kernel_regularizer=regularizer,
                        bias_regularizer=regularizer
                    )
                )

        model.add(
            tf.keras.layers.Dense(
                self.output_size,
                activation="linear",
                kernel_regularizer=regularizer,
                bias_regularizer=regularizer
            )
        )
        model.build((None, TIME_STEPS, len(PREDICTORS)))

        model.load_weights(Wf)
        self.models.append(model)


    def BuildEnsemble(self, ModelsList):
        for (arch, Wf, r) in ModelsList:
            if isinstance(arch, str):
                arch = ast.literal_eval(arch)  
            self.SetModels(arch, Wf, r)


    def PredictModel(self, model, x):

        y_pred = model.predict(x)
        y_pred = self.out_scaler.inverse_transform(y_pred)
        init_vals = np.array([Datasets[i][name].iloc[0] for name in self.target_int])
        
        for j in range(self.output_size):
            y_pred[:, j] = init_vals[j] + np.cumsum(y_pred[:, j] * TS)
        return y_pred


    def EvalEnsemble(self, datasets, Wensemble):

        for d, dataset in enumerate(datasets):

            x, y = CreateSequences(
                dataset[self.predictors],
                dataset[self.targets],
                TIME_STEPS
            )

            y_true = dataset[self.target_int].iloc[TIME_STEPS:].values

            y_ensemble = np.zeros((len(x), self.output_size))

            print(f"\n===== DATASET: {SETS[d]} =====")

            # 🔹 métricas individuais
            for i, model in enumerate(self.models):

                y_pred = self.PredictModel(model, x)

                y_ensemble += Wensemble[i] * y_pred

                for j, name in enumerate(self.target_int):

                    r2 = r2_score(y_true[:, j], y_pred[:, j])
                    mse = mean_squared_error(y_true[:, j], y_pred[:, j])

                    print(
                        f"Model {i+1} | {name} -> "
                        f"R² = {r2:.4f}, MSE = {mse:.4e}"
                    )

            # 🔹 métricas do ensemble
            print("---- ENSEMBLE ----")

            for j, name in enumerate(self.target_int):

                r2 = r2_score(y_true[:, j], y_ensemble[:, j])
                mse = mean_squared_error(y_true[:, j], y_ensemble[:, j])

                print(
                    f"Ensemble | {name} -> "
                    f"R² = {r2:.4f}, MSE = {mse:.4e}"
                )

In [11]:
def Test(predictors, targets, int_targets, out_scaler, i):
    ModelsList = []

    for _, row in models[i].iterrows():

        # garante que arch é lista
        arch = row["Neurons"]
        r = float(row["reg"])
        weights_path = f".\Re-Run\{int_targets[0]}\Ts-03\weights\{row['model']}.weights.h5"

        ModelsList.append((arch, weights_path, r))
        
    Wensemble = np.ones(len(ModelsList)) / len(ModelsList) 
    
    ensemble = EnsembleNets(
        predictors=predictors,
        targets=targets,
        int_targets=int_targets,
        out_scaler=out_scaler,
    )

    ensemble.BuildEnsemble(ModelsList)

    ensemble.EvalEnsemble(
        datasets=NormDatasets,
        Wensemble=Wensemble,
    )

In [12]:
Test(predictors=["PwmD", "PwmE"],
     targets=["DeltaTheta"],
     int_targets=["Theta"],
     out_scaler=OUT_SCALERS["DeltaTheta"], 
     i=0)

ValueError: Layer 'simple_rnn_cell' expected 3 variables, but received 0 variables during loading. Expected: ['simple_rnn/simple_rnn_cell/kernel:0', 'simple_rnn/simple_rnn_cell/recurrent_kernel:0', 'simple_rnn/simple_rnn_cell/bias:0']

In [ ]:
Test(predictors=["PwmD", "PwmE", "Theta"],
     targets=["DeltaX"],
     int_targets=["X"],
     out_scaler=OUT_SCALERS["DeltaX"], 
     i=1)


===== DATASET: Train_1 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | X -> R² = 0.8402, MSE = 1.7237e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.8402, MSE = 1.7237e-02

===== DATASET: Train_2 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | X -> R² = 0.8408, MSE = 1.3413e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.8408, MSE = 1.3413e-02

===== DATASET: Test_1 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | X -> R² = 0.8850, MSE = 1.2051e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.8850, MSE = 1.2051e-02

===== DATASET: Test_2 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | X -> R² = 0.8313, MSE = 1.3880e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.8313, MSE = 1.3880e-02

===== DATASET: Test_3 =====
45/45 [==============================] - 0s 2ms/step
Model 1 | X -> R² = 0.3348, MSE = 6.1217e-02
---- ENSEMBLE ----
Ensemble | X -> R² = 0.3348, MSE = 6.1217e-02

===== DATASET: Val =====
40/40 [=====

In [ ]:
Test(predictors=["PwmD", "PwmE", "Theta"],
     targets=["DeltaY"],
     int_targets=["Y"],
     out_scaler=OUT_SCALERS["DeltaY"], 
     i=2)


===== DATASET: Train_1 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | Y -> R² = 0.7760, MSE = 1.8771e-02
31/31 [==============================] - 0s 1ms/step
Model 2 | Y -> R² = 0.6005, MSE = 3.3479e-02
31/31 [==============================] - 0s 2ms/step
Model 3 | Y -> R² = 0.7489, MSE = 2.1042e-02
---- ENSEMBLE ----
Ensemble | Y -> R² = 0.7515, MSE = 2.0829e-02

===== DATASET: Train_2 =====
31/31 [==============================] - 0s 2ms/step
Model 1 | Y -> R² = 0.9473, MSE = 5.6734e-03
31/31 [==============================] - 0s 1ms/step
Model 2 | Y -> R² = 0.9602, MSE = 4.2813e-03
31/31 [==============================] - 0s 2ms/step
Model 3 | Y -> R² = 0.9335, MSE = 7.1555e-03
---- ENSEMBLE ----
Ensemble | Y -> R² = 0.9912, MSE = 9.4873e-04

===== DATASET: Test_1 =====
31/31 [==============================] - 0s 1ms/step
Model 1 | Y -> R² = 0.7277, MSE = 2.2321e-02
31/31 [==============================] - 0s 1ms/step
Model 2 | Y -> R² = 0.5368, MSE = 3.7969e-